In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "0,1,3,4"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader


KeyboardInterrupt



In [2]:
model_path = "/mnt/petrelfs/share_data/songmingyang/model/reasoning/policy_models/DeepSeek-R1-Distill-Qwen-7B"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [29]:
train_data_path = "deepscaler/dataset/train.parquet"
train_dataset = RLHFDataset(parquet_files=train_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')
train_dataloader = DataLoader(dataset=train_dataset,
                            batch_size=2,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 40315
filter dataset len: 40306


In [47]:
"""simple-RL 8k math train data"""
import pandas as pd
import json

json_path = "/mnt/petrelfs/huzican/R1/simpleRL-reason/train/data/math_level3to5_data_processed_with_qwen_prompt.json"
train_path = "dataset/train.parquet"

with open(json_path, 'r') as f:
    data = json.load(f)

converted_data = []
for idx, item in enumerate(data):
    converted_item = {
        'data_source': 'math',
        'prompt': [{
            'content': item['question'],  # JSON中的question字段作为prompt的content
            'role': 'user'
        }],
        'ability': 'math',
        'reward_model': {
            'ground_truth': item['ground_truth_answer'],
            'style': 'rule'
        },
        'extra_info': {
            'index': idx, 
            'split': 'train'  
        }
    }
    converted_data.append(converted_item)

df = pd.DataFrame(converted_data)

df.to_parquet(train_path)

In [48]:
"""valid_data aime*8+math 500"""
import pandas as pd
valid_data_path = "/mnt/petrelfs/share_data/yanjianhao/valid_v2/valid.parquet"
test_path = "dataset/test.parquet"
df = pd.read_parquet(valid_data_path)
new_data = []
for _, row in df.iterrows():
    row_dict = row.to_dict()
    # 更新prompt，只保留user内容
    row_dict['prompt'] = [{
        'content': row_dict['prompt'][1]['content'],
        'role': 'user'
    }]
    # 更新split
    row_dict['extra_info']['split'] = 'test'
    new_data.append(row_dict)

new_df = pd.DataFrame(new_data)
new_df.to_parquet(test_path)
print(row_dict)

{'data_source': 'math', 'prompt': [{'content': 'Altitudes $\\overline{AD}$ and $\\overline{BE}$ of $\\triangle ABC$ intersect at $H$.  If $\\angle BAC = 54^\\circ$ and $\\angle ABC = 52^\\circ$, then what is $\\angle AHB$?', 'role': 'user'}], 'ability': 'math', 'reward_model': {'ground_truth': '106^\\circ', 'style': 'rule'}, 'extra_info': {'index': 499, 'split': 'test'}}


In [45]:
import pandas as pd
train_path = "/mnt/petrelfs/huzican/R1/RLKD_RM/dataset/train.parquet"
df = pd.read_parquet(train_path)
# print(df.iloc[0].keys())
print(len(df))
# print(df.iloc[1])
for key, value in df.iloc[1].items():
    print(f"{key}: {value}")

8523
data_source: math
prompt: {'content': 'For how many integer values of $x$ is $5x^{2}+19x+16 > 20$ not satisfied?', 'role': 'user'}
ability: math
reward_model: {'ground_truth': '5', 'style': 'rule'}
extra_info: {'index': 1, 'split': 'train'}


In [27]:
import pandas as pd
df = pd.read_parquet('deepscaler/dataset/aime.parquet')
# print(df.iloc[0].keys())
print(len(df))
print(df.iloc[1])
for key, value in df.iloc[1].items():
    print(f"{key}: {value}")

30
data_source                                                      
prompt          [{'content': 'There exist real numbers $x$ and...
ability                                                      math
reward_model             {'ground_truth': '025', 'style': 'rule'}
extra_info                          {'index': 1, 'split': 'test'}
Name: 1, dtype: object
data_source: 
prompt: [{'content': "There exist real numbers $x$ and $y$, both greater than 1, such that $\\log_x\\left(y^x\\right)=\\log_y\\left(x^{4y}\\right)=10$. Find $xy$. Let's think step by step and output the final answer within \\boxed{}.", 'role': 'user'}]
ability: math
reward_model: {'ground_truth': '025', 'style': 'rule'}
extra_info: {'index': 1, 'split': 'test'}


In [25]:
import pandas as pd
df = pd.read_parquet('/mnt/petrelfs/share_data/yanjianhao/valid_v2/valid.parquet')
print(len(df))
# print(df.iloc[0].keys())
for key, value in df.iloc[0].items():
    print(f"{key}: {value}")
# for key, value in df.iloc[9].items():
#     print(f"{key}: {value}")

740
data_source: aime
prompt: [{'content': 'Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: "<think>\n {thoughts} </think>\n". Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After "</think>\n," in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions.', 'role': 'system'}
 {'content': 'Every morning Aya goes for a $9$-kilometer-long walk and stops at a coffee shop afterwards. When she walks at a constant speed of $s$ kilometers per hour, 

In [19]:
import pandas as pd
df = pd.read_parquet('/mnt/petrelfs/share_data/yanjianhao/open-r1-data-spec-v4-filter/train.parquet')
print(df.iloc[0].keys())
for key, value in df.iloc[0].items():
    print(f"{key}: {value}")
# for key, value in df.iloc[2].items():
#     print(f"{key}: {value}")
# for key, value in df.iloc[3].items():
#     print(f"{key}: {value}")

Index(['data_source', 'prompt', 'target', 'ability', 'reward_model',
       'extra_info'],
      dtype='object')
data_source: 
prompt: [{'content': 'Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: "<think>\n {thoughts} </think>\n". Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After "</think>\n," in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions.', 'role': 'system'}
 {'content': '## Task B-1.3.\n\nA ship traveling along a river h

In [27]:
val_data_path = "/mnt/petrelfs/share_data/yanjianhao/open-r1-data-spec-v4-filter/math.parquet"
val_dataset = RLHFDataset(parquet_files=val_data_path,
                          tokenizer=tokenizer,
                          prompt_key='prompt',
                          max_prompt_length=1024,
                          filter_prompts=True,
                          return_raw_chat=False,
                          truncation='error')
val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=len(val_dataset),
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 500
filter dataset len: 500


In [28]:
print(val_dataset.__getitem__(1))

{'data_source': '', 'ability': 'math', 'reward_model': {'ground_truth': 'p - q', 'style': 'rule'}, 'extra_info': {'index': 1, 'split': 'default'}, 'input_ids': tensor([151643, 151643, 151643,  ...,     80,   2418, 151645]), 'attention_mask': tensor([0, 0, 0,  ..., 1, 1, 1]), 'position_ids': tensor([  0,   0,   0,  ..., 251, 252, 253]), 'index': 1}


In [16]:
print(train_dataset.__getitem__(13494).keys())
print(train_dataset.__getitem__(13494)['target'])

dict_keys(['data_source', 'target', 'ability', 'reward_model', 'extra_info', 'input_ids', 'attention_mask', 'position_ids', 'index'])
[{'content': '<think>\nOkay, let\'s see. I have two parts here, a) and b). Both are about selecting students from a class of 30. Let me start with part a).\n\nPart a) says: "From a class of 30 students, two students need to be selected to participate in a mathematics olympiad. In how many ways can this be done?"\n\nHmm. So, I need to figure out how many different pairs of students can be chosen from 30. This sounds like a combination problem because the order in which we select the students doesn\'t matter. For example, selecting Alice first and Bob second is the same as selecting Bob first and Alice second—they’re both the same team. So combinations are used when the order doesn\'t matter, and permutations when it does. Since the problem is about selecting participants without any mention of order, combinations should be the right approach.\n\nThe formu

In [18]:
for batch in train_dataloader:
    

{'input_ids': tensor([[151643, 151643, 151643,  ...,      3,     30, 151645],
        [151643, 151643, 151643,  ...,  47679,     13, 151645]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]]), 'position_ids': tensor([[  0,   0,   0,  ..., 267, 268, 269],
        [  0,   0,   0,  ..., 212, 213, 214]]), 'data_source': array(['', ''], dtype=object), 'target': array([[{'content': "<think>\nOkay, let's see. We have two points, A and B, 308 meters apart. A point starts moving from A towards B. In the first second, it goes 15 meters, then each next second it goes 1 meter less. Another point starts moving from B towards A, but it starts 3 seconds after the first one. This second point goes 20 meters in the first second, then each subsequent second it goes 3 meters more than before. We need to find out where they meet relative to point A.\n\nAlright, let's break this down. Let's call the first point Point 1 (starting from A) and the second point Point 2 (s